In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import numpy as np
import pandas as pd
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))


In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics paddleocr paddlepaddle
print("\n✅ Cài xong.")


In [ ]:
import torch

# ⭐ Một số GPU Kaggle gán cho session (vd P100 đời cũ) không còn được bản torch mới nhất hỗ
#   trợ -> lỗi "CUDA error: no kernel image is available for execution on the device" dù
#   nvidia-smi vẫn thấy card. Cell này tự thử chạy 1 phép tính nhỏ trên GPU, nếu lỗi thì tự
#   động chuyển toàn bộ notebook sang chạy CPU (chậm hơn nhưng không crash) thay vì phải tự
#   sửa tay mỗi lần đổi GPU.
def pick_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        (torch.zeros(1) + torch.zeros(1)).cuda()
        return 0
    except RuntimeError as e:
        print("⚠️ GPU không tương thích với bản torch hiện tại, chuyển sang CPU:", e)
        return 'cpu'

DEVICE = pick_device()
print("✅ Dùng device:", DEVICE)


## 👉 ĐIỀN VÀO ĐÂY: đường dẫn `best.pt`, ảnh test, video test

Chỉ sửa các dòng trong cell ngay dưới đây, không cần đụng gì khác.

Lấy đường dẫn ở đâu: `best.pt`, ảnh, video đều phải được "Add Data" (Add Input) lên Kaggle trước
(Upload), sau đó **chạy cell đầu tiên của notebook** (`os.walk('/kaggle/input')`) — nó in ra chính
xác từng đường dẫn file đang có. Copy đúng dòng đó dán vào bên dưới.

In [ ]:
# ⬇️ 0) Đã có sẵn best.pt chưa? True = dùng luôn best.pt bên dưới, KHÔNG train (Run All sẽ tự bỏ qua phần train).
#       False = chưa có, muốn train mới từ dataset (cần điền DATA_YAML ở phần "Dataset để train" bên dưới).
DA_CO_BEST_PT = True

# ⬇️ 1) Đường dẫn tới file best.pt (chỉ cần đúng khi DA_CO_BEST_PT = True)
BEST_PLATE_WEIGHTS = '/kaggle/input/ten-dataset-cua-ban/best.pt'

# ⬇️ 2) Đường dẫn ảnh test (thêm bao nhiêu dòng cũng được)
TEST_IMAGES = [
    '/kaggle/input/ten-dataset-cua-ban/anh1.jpg',
    '/kaggle/input/ten-dataset-cua-ban/anh2.jpg',
]

# ⬇️ 3) Đường dẫn video test (thay cho '111.mp4' trong code gốc của bạn)
VIDEO_PATH = '/kaggle/input/ten-dataset-cua-ban/video.mp4'
OUTPUT_VIDEO = '/kaggle/working/result_video.mp4'


## Dataset để train (chỉ cần nếu **chưa có** `best.pt`)

Nếu ở trên bạn đã điền đúng `BEST_PLATE_WEIGHTS`, **bỏ qua toàn bộ phần train này**, chạy thẳng
xuống phần "Test detect + đọc ký tự" ở dưới.

Nếu chưa có `best.pt` và muốn train mới, cần một dataset detect biển số ở định dạng YOLOv8
(thư mục `train/images`, `train/labels`... kèm file `data.yaml`), upload qua "Add Data" giống ảnh/weights ở trên.

In [ ]:
import glob

# ⬇️ Chỉ cần điền nếu train mới (không có best.pt): đường dẫn tới data.yaml của dataset
DATA_YAML = '/kaggle/input/ten-dataset-cua-ban/data.yaml'


### Xem thử vài ảnh + nhãn trước khi train

Kiểm tra nhanh vài ảnh cùng bounding box đã gán nhãn, để chắc dataset đúng (khung bao quanh đúng
biển số) trước khi tốn thời gian train. Bỏ qua cell này nếu đã có `best.pt`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import yaml

def parse_label_line(parts, w, h):
    """Trả về (cls, x1, y1, x2, y2) dù dòng là bbox (5 số) hay polygon (>5 số)."""
    cls = int(float(parts[0]))
    vals = list(map(float, parts[1:]))
    if len(vals) == 4:
        xc, yc, bw, bh = vals
        x1 = (xc - bw / 2) * w; y1 = (yc - bh / 2) * h
        x2 = (xc + bw / 2) * w; y2 = (yc + bh / 2) * h
    else:
        xs = [vals[i] * w for i in range(0, len(vals), 2)]
        ys = [vals[i] * h for i in range(1, len(vals), 2)]
        x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
    return cls, int(x1), int(y1), int(x2), int(y2)

if DA_CO_BEST_PT:
    print("⏭️ DA_CO_BEST_PT = True — đã có best.pt, bỏ qua preview dataset.")
else:
    with open(DATA_YAML) as f:
        data_cfg = yaml.safe_load(f)
    print("Số lớp:", data_cfg.get('nc'))
    print("Danh sách lớp:", data_cfg.get('names'))
    names = data_cfg.get('names')

    train_img_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'images')
    train_lbl_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'labels')
    sample_imgs = sorted(glob.glob(os.path.join(train_img_dir, '*')))[:6]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, img_path in zip(axes.flat, sample_imgs):
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        lbl_path = os.path.join(train_lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + '.txt')
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.split()
                    if not parts:
                        continue
                    cls, x1, y1, x2, y2 = parse_label_line(parts, w, h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 2)
                    label = names[cls] if cls < len(names) else str(cls)
                    cv2.putText(img, label, (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        ax.imshow(img)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


### Train YOLOv8 detect vị trí biển số

Chỉ detect **khung bao (bounding box)** của biển số trong ảnh, không đọc ký tự — đọc ký tự nằm ở
phần OCR bên dưới. Bỏ qua cell này nếu đã có `best.pt`.

In [ ]:
from ultralytics import YOLO

if DA_CO_BEST_PT:
    print("⏭️ DA_CO_BEST_PT = True — đã có best.pt, bỏ qua train:", BEST_PLATE_WEIGHTS)
else:
    plate_model = YOLO('yolov8n.pt')

    plate_model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,           # ⭐ ảnh cả xe/khung hình lớn hơn nhiều so với detect ký tự -> cần imgsz lớn
        batch=16,
        patience=20,         # ⭐ early stop nếu không cải thiện sau 20 epoch
        device=DEVICE,
        project='/kaggle/working/yolo_plate_runs',
        name='train',
    )

    # ⭐ train() trả về kiểu khác nhau tuỳ version ultralytics (có bản trả dict,
    #   không có .save_dir) -> lấy qua plate_model.trainer.save_dir cho ổn định
    BEST_PLATE_WEIGHTS = str(plate_model.trainer.save_dir / 'weights' / 'best.pt')
    print("✅ Train xong, weights tốt nhất tại:", BEST_PLATE_WEIGHTS)


## Test detect + đọc ký tự trên ảnh (`inp` = ảnh input)

Cùng logic y hệt code video bên dưới, chỉ khác `inp` là 1 ảnh (đọc bằng `cv2.imread`) thay vì
từng `frame` lấy ra từ `cap.read()`: YOLO detect biển số trên `inp` → crop → PaddleOCR đọc chữ →
vẽ khung + chữ lên `inp` → hiện ảnh kết quả.

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR
import cv2
import matplotlib.pyplot as plt

# Model YOLO đã train để detect biển số
yolo_model = YOLO(BEST_PLATE_WEIGHTS)

ocr = PaddleOCR(
    use_textline_orientation=True,
    lang='en',
    enable_mkldnn=False
)

def recognize_plate(plate_img):
    if plate_img is None or plate_img.size == 0:
        return ""
    result = ocr.predict(plate_img)
    texts = []
    for res in result:
        texts.extend(res['rec_texts'])
    return " ".join(texts)

for IMG_PATH in TEST_IMAGES:
    inp = cv2.imread(IMG_PATH)  # ⬅️ ảnh input (giống "frame" bên code video, nhưng chỉ 1 ảnh)

    yolo_results = yolo_model(inp, device=DEVICE)[0]

    for box in yolo_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        plate_crop = inp[y1:y2, x1:x2]

        text = recognize_plate(plate_crop)

        cv2.rectangle(inp, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(inp, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    # ⭐ Kaggle không có màn hình -> không dùng cv2.imshow() được, hiện ảnh bằng matplotlib.
    plt.figure(figsize=(8, 6))
    plt.imshow(cv2.cvtColor(inp, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(IMG_PATH))
    plt.axis('off')
    plt.show()


## Đọc ký tự biển số từ video — đúng code bạn gửi

Y hệt logic bạn đưa: YOLO detect biển số trên từng khung hình → crop → đưa cho PaddleOCR đọc chữ
→ vẽ khung + chữ lên frame. Chỉ đổi **đúng 3 chỗ bắt buộc** để chạy được trên Kaggle (không đổi gì
khác so với code gốc):

1. `YOLO('best.pt')` → `YOLO(BEST_PLATE_WEIGHTS)` (dùng đường dẫn bạn đã điền ở cell "ĐIỀN VÀO ĐÂY" thay vì gõ chết tên file).
2. `cv2.VideoCapture('111.mp4')` → `cv2.VideoCapture(VIDEO_PATH)` (tương tự, dùng đường dẫn đã điền).
3. `cv2.imshow()` / `cv2.waitKey()` / `cv2.destroyAllWindows()` → `cv2.VideoWriter` ghi ra file
   `OUTPUT_VIDEO`, vì Kaggle chạy trên server, **không có màn hình** để mở cửa sổ — giữ nguyên
   `cv2.imshow()` sẽ báo lỗi ngay khi chạy.

**Thêm 1 phần: cache kết quả OCR giữa các frame.** Cùng 1 biển số thường xuất hiện liên tục ở
nhiều frame liền nhau, gần như cùng 1 vị trí. Nếu frame nào cũng chạy PaddleOCR lại từ đầu thì vừa
chậm vừa dễ ra chữ khác nhau giữa các frame (chớp nháy). Nên mỗi frame, trước khi OCR, so khung
hiện tại với các khung đã nhận diện ở frame ngay trước (bằng IoU — độ chồng lấp 2 khung): nếu
trùng vị trí (cùng 1 biển, chỉ là xe/camera hơi rung nên toạ độ lệch chút ít) thì **lấy lại chữ đã
đọc trước đó**, chỉ OCR lại khi là biển mới chưa từng thấy ở frame trước.

In [ ]:
from ultralytics import YOLO
from paddleocr import PaddleOCR
import cv2

# Model YOLO đã train để detect biển số
yolo_model = YOLO(BEST_PLATE_WEIGHTS)

ocr = PaddleOCR(
    use_textline_orientation=True,
    lang='en',
    enable_mkldnn=False
)

def recognize_plate(plate_img):
    if plate_img is None or plate_img.size == 0:
        return ""
    result = ocr.predict(plate_img)
    texts = []
    for res in result:
        texts.extend(res['rec_texts'])
    return " ".join(texts)

def iou(box_a, box_b):
    """Độ chồng lấp giữa 2 khung (x1, y1, x2, y2), 0 = không chạm nhau, 1 = trùng khít."""
    xa1, ya1, xa2, ya2 = box_a
    xb1, yb1, xb2, yb2 = box_b
    inter_x1, inter_y1 = max(xa1, xb1), max(ya1, yb1)
    inter_x2, inter_y2 = min(xa2, xb2), min(ya2, yb2)
    inter = max(0, inter_x2 - inter_x1) * max(0, inter_y2 - inter_y1)
    area_a = max(0, xa2 - xa1) * max(0, ya2 - ya1)
    area_b = max(0, xb2 - xb1) * max(0, yb2 - yb1)
    return inter / (area_a + area_b - inter + 1e-6)

IOU_MATCH_THRESHOLD = 0.5   # ⭐ 2 khung được coi là "cùng 1 biển" giữa 2 frame nếu chồng lấp >= 50%

cap = cv2.VideoCapture(VIDEO_PATH)  # ⬅️ đổi từ '111.mp4' sang biến VIDEO_PATH

# ⭐ Kaggle không có màn hình -> không dùng cv2.imshow()/cv2.waitKey() được, ghi video kết quả
#   ra file bằng VideoWriter thay vì mở cửa sổ.
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
fps = cap.get(cv2.CAP_PROP_FPS) or 25
frame_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (frame_w, frame_h))

# ⭐ Danh sách biển đã đọc được ở frame TRƯỚC: [{'box': (x1,y1,x2,y2), 'text': str}, ...]
#   Dùng để so khung frame này với frame trước, trùng vị trí thì lấy lại chữ cũ, khỏi OCR lại.
tracked_plates = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    yolo_results = yolo_model(frame, device=DEVICE)[0]

    new_tracked = []
    for box in yolo_results.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cur_box = (x1, y1, x2, y2)

        # ⭐ So với các khung đã nhận diện ở frame trước -> trùng vị trí (IoU cao) thì lấy lại
        #   chữ đã đọc trước đó thay vì OCR lại từ đầu.
        text = None
        for tp in tracked_plates:
            if iou(cur_box, tp['box']) >= IOU_MATCH_THRESHOLD:
                text = tp['text']
                break

        if text is None:
            plate_crop = frame[y1:y2, x1:x2]
            text = recognize_plate(plate_crop)

        new_tracked.append({'box': cur_box, 'text': text})

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    tracked_plates = new_tracked   # ⭐ cập nhật để làm cơ sở so sánh cho frame kế tiếp

    writer.write(frame)  # ⬅️ thay cho cv2.imshow(...) + cv2.waitKey(1)

cap.release()
writer.release()  # ⬅️ thay cho cv2.destroyAllWindows()
print(f"✅ Xong, video kết quả lưu tại: {OUTPUT_VIDEO}")


### Xem lại video kết quả

Phát video đã ghi ngay trong notebook (thay cho cửa sổ `cv2.imshow` không dùng được trên Kaggle).

In [ ]:
from IPython.display import Video

Video(OUTPUT_VIDEO, embed=True, width=640)
